# CAIM Lab Session 2: Intro to ElasticSearch

In this session you will learn:

- a few basics on the `ElasticSearch` database
- how to index a set of documents and how to ask simple queries about these documents
- how to do this from `Python`
- based on the previous, you will compute the boolean and tf-idf matrix for the toy corpus used in class

## 1. ElasticSearch

[ElasticSearch](https://www.elastic.co/) is a _NoSQL/document_ database with the capability of indexing and searching text documents. As a rough analogue, we can use the following table for the equivalence between ElasticSearch and a more classical relational database:

| Relational DB | ElasticSearch |
|---|---|
| Database | Index |
| Row / record | Document |
| Column | Field |

An index can be thought of as an optimized collection of documents and each document is a collection of fields, which are the key-value pairs that contain your data.

`ElasticSearch` is a pretty big beast with many options. Luckily, there is much documentation, a few useful links are:

- Here is the [full documentation](https://www.elastic.co/guide/en/elasticsearch/reference/current/index.html)
- Intros you may want to have a look at:
    - https://medium.com/expedia-group-tech/getting-started-with-elastic-search-6af62d7df8dd
    - http://joelabrahamsson.com/elasticsearch-101
- You found another one that you liked? Let us know.

## 2. Running ElasticSearch

This database runs as a web service in a machine and can be accessed using a REST
web API.

The ElasticSearch binaries are in `/opt/elasticsearch-8.2.2/`.

Depending on the disk space that you have available you can run directly the script that starts the
database, so all the data will be stored in your user directory, or you can change the configuration
of the database in order to use the space in `/tmp`. 
Also, security needs to be disabled (through the `xpack.security.enabled` configuration option) so make
sure the line is found in the configuration file.

```bash
cp -r /opt/elasticsearch-8.2.2/config/ /tmp
```

Modify or add the following lines to `/tmp/config/elasticsearch.yml`:

```
path.data : /tmp/elastic_data
path.logs : /tmp/elastic_logs
xpack.security.enabled : false
xpack.security.enrollment.enabled : false
```

Set the environment variable `ES_PATH_CONF` to point to the configuration files. Example (if using `tcsh`):

```bash
setenv ES_PATH_CONF /tmp/config
```

Now you can run ElasticSearch with:

```bash
/opt/elasticsearch-8.2.2/bin/elasticsearch
```

After a few seconds (and a lot of logging) the database will be up and running; you may need to hit return for the prompt to show up. To test whether `ElasticSearch` is working execute the code in the cell below. __The database needs to be running throughout the execution of this script, otherwise you will get a connection error.__

In [2]:
from pprint import pprint
import requests

try:
    resp = requests.get('http://localhost:9200/')
    pprint(resp.content)

except Exception:
    print('elasticsearch is not running')

elasticsearch is not running


If `ElasticSearch` is working you will see an answer from the server; otherwise you will see a message indicating that it is not running. You can try also throwing the URL http://localhost:9200 to your browser; you should get a similar answer.

## 3. Indexing and querying

`ElasticSearch` is a database that allows storing documents (tables do not need a predefined schema as in relational databases). Text in these documents can be processed so the queries extend beyond exact matches allowing complex queries, fuzzy matching and ranking documents respect to the actual match.

These kinds of databases are behind search engines like Google Search or Bing.

There are different ways of operating with ElasticSearch. It is deployed esentially as a web service with a REST API, so it can be accessed basically from any language with a library for operating with HTTP servers.

We are going to use two python libraries for programming on top of ElasticSearch: `elasticsearch` and `elasticsearch-dsl`. Both provide access to ElasticSearch functionalities hiding and making more programming-friendly the interactions, the second one is more convenient for configurating and searching. Make sure both python libraries are installed to proceed with this session.

In [6]:
!pip3 install elasticsearch --user
!pip3 install elasticsearch-dsl --user

We are only going to see the essential elements for developing the session but feel free to learn more.

To interact with ElasticSearch with need a client object of type `Elasticsearch`.

In [7]:
from elasticsearch import Elasticsearch

client = Elasticsearch("http://localhost:9200", request_timeout=1000)

With this client you have a connection for operating with Elastic search. Now we will create an index. There are index operations in each library, but the one in `elasticseach-dsl` is simpler to use.

In [8]:
from elasticsearch_dsl import Index

index = Index('test', using=client)  # if it does not exist, it is created; if it does exist, then it connects

First we will need some text to index, for testing purposes we are going to use the python library `loremipsum`. We will need to install it first if it is not installed already, uncomment the code in next cell if you need to install the library

In [9]:
!pip3 install lorem --user  # Restart the kernel if you are not able to import the library in the next cell

Now we create some random paragraphs

In [10]:
import lorem

texts = [lorem.paragraph() for _ in range(10)]
print(len(texts))
print(texts[0])

10
Labore voluptatem modi sed modi. Etincidunt ipsum modi quaerat adipisci est. Dolorem dolore ipsum porro voluptatem dolorem. Eius aliquam adipisci numquam dolor quiquia. Dolore eius voluptatem voluptatem porro. Quiquia etincidunt non eius. Non consectetur ipsum numquam amet dolore ut labore. Dolorem voluptatem etincidunt quiquia quiquia numquam quaerat porro.


Now we can index the paragraphs in ElasticSearch using the `index` method. The document is passed as a python dictionary with the `document` parameter. The keys of the dictionary will be the fields of the document, in this case we well have only one (`text`) -- here, we use this tag but could use anything we wanted to.

In [11]:
for t in texts:
    client.index(index='test', document={'text': t})
    print(f'Indexing new text: {t[:70]} ...')
client.indices.refresh(index='test')

Indexing new text: Labore voluptatem modi sed modi. Etincidunt ipsum modi quaerat adipisc ...
Indexing new text: Etincidunt sed ut ut ipsum dolorem numquam. Sed est tempora numquam do ...
Indexing new text: Dolorem numquam quaerat sit quaerat quisquam. Velit dolor neque velit  ...
Indexing new text: Neque consectetur dolorem adipisci dolor magnam. Tempora aliquam numqu ...
Indexing new text: Dolorem velit sit labore. Dolorem modi quiquia dolorem numquam volupta ...
Indexing new text: Adipisci quaerat aliquam porro. Non quisquam numquam quisquam. Porro s ...
Indexing new text: Dolorem eius amet sed quisquam quiquia amet est. Tempora etincidunt am ...
Indexing new text: Ut non sit quiquia est. Amet sed amet ut est numquam. Ipsum sit adipis ...
Indexing new text: Porro ut est ut etincidunt aliquam ipsum. Consectetur tempora quiquia  ...
Indexing new text: Quisquam modi sed quiquia modi non eius. Ipsum amet neque tempora. Eiu ...


ObjectApiResponse({'_shards': {'total': 2, 'successful': 1, 'failed': 0}})

In case we want to get all docs in the index, we can do the following:

In [12]:
# get all docs in index 'test'
resp = client.search(index="test", query={"match_all": {}})

# print them
print(f"Got {resp['hits']['total']['value']} hits:")
for hit in resp['hits']['hits']:
    pprint(hit["_source"])

Got 10 hits:
{'text': 'Labore voluptatem modi sed modi. Etincidunt ipsum modi quaerat '
         'adipisci est. Dolorem dolore ipsum porro voluptatem dolorem. Eius '
         'aliquam adipisci numquam dolor quiquia. Dolore eius voluptatem '
         'voluptatem porro. Quiquia etincidunt non eius. Non consectetur ipsum '
         'numquam amet dolore ut labore. Dolorem voluptatem etincidunt quiquia '
         'quiquia numquam quaerat porro.'}
{'text': 'Etincidunt sed ut ut ipsum dolorem numquam. Sed est tempora numquam '
         'dolor. Amet adipisci velit quaerat dolor neque. Quiquia sit est non '
         'dolor est sit. Dolor numquam etincidunt dolore.'}
{'text': 'Dolorem numquam quaerat sit quaerat quisquam. Velit dolor neque '
         'velit dolorem porro eius. Magnam est tempora aliquam adipisci sit. '
         'Dolore eius aliquam magnam porro magnam. Sit neque dolorem labore. '
         'Quiquia magnam dolorem quaerat modi labore. Eius ut etincidunt '
         'dolorem non qua

We can also search for documents that contain a given keyword:

In [13]:
from elasticsearch_dsl import Search

# the following search query specifies the field where we want to search
s_obj = Search(using=client, index='test')
sq = s_obj.query('match', text='non')
resp = sq.execute()

print(f'Found {len(resp)} matches.')

for hit in resp:
    print(f'\nID: {hit.meta.id}\nText: {hit.text}')

Found 7 matches.

ID: D2Juv5kB9jejAi1w_DtF
Text: Neque consectetur dolorem adipisci dolor magnam. Tempora aliquam numquam labore quiquia dolore. Aliquam amet voluptatem modi quisquam quaerat dolor. Sit dolor neque consectetur voluptatem sed labore. Ipsum modi amet amet sit amet sit modi. Magnam neque est non non dolorem sit neque. Quisquam voluptatem ipsum numquam non aliquam. Quisquam quaerat dolorem non. Velit aliquam adipisci numquam dolore. Neque magnam numquam sed.

ID: EWJuv5kB9jejAi1w_Du5
Text: Adipisci quaerat aliquam porro. Non quisquam numquam quisquam. Porro sed neque consectetur amet numquam sit. Quiquia est neque sit etincidunt labore sit. Quaerat quisquam etincidunt non dolore numquam. Dolorem quisquam amet quisquam. Adipisci voluptatem ipsum ut quiquia dolore est numquam.

ID: FWJuv5kB9jejAi1w_Tuz
Text: Quisquam modi sed quiquia modi non eius. Ipsum amet neque tempora. Eius sed porro voluptatem consectetur tempora. Porro dolorem eius sed etincidunt numquam velit consecte

## 4. Counting words and docs

`Elastic search` helps us to obtain the counts of words in each document. For example, the following code obtains the counts of words of a whole index by adding the counts of words obtained from each document through the functionality of `termvectors`. This function also allows us to get _document counts_ for computing tf-idf weights, by setting the `term_statistics` option to `True`.

In [14]:
from elasticsearch.helpers import scan
from collections import Counter

# Search for all the documents and query the list of (word, frequency) of each one
# Totals are accumulated using a Counter for term frequencies
word_counts = Counter()
sc = scan(client, index='test', query={"query" : {"match_all": {}}})
for s in sc:
    tv = client.termvectors(index='test', id=s['_id'], fields=['text'], term_statistics=True, positions=False)
    if 'text' in tv['term_vectors']:   # just in case some document has no field named 'text'
        for t in tv['term_vectors']['text']['terms']:
            word = t
            count = tv['term_vectors']['text']['terms'][t]['term_freq']
            word_counts.update({word: count})


In [15]:
# show word frequencies
word_counts.most_common()

[('dolorem', 25),
 ('numquam', 24),
 ('amet', 23),
 ('sit', 22),
 ('quiquia', 21),
 ('voluptatem', 20),
 ('quisquam', 19),
 ('eius', 18),
 ('sed', 18),
 ('est', 17),
 ('porro', 17),
 ('ut', 17),
 ('neque', 17),
 ('magnam', 17),
 ('etincidunt', 16),
 ('ipsum', 16),
 ('tempora', 16),
 ('aliquam', 15),
 ('consectetur', 15),
 ('non', 15),
 ('velit', 15),
 ('adipisci', 14),
 ('modi', 14),
 ('dolor', 13),
 ('labore', 13),
 ('quaerat', 13),
 ('dolore', 12)]

## 5. Proposed simple exercise

To get more familiar with elasticsearch, we propose that you _generate the Boolean and tf-idf matrices_ for the toy example that we used in class. You will find 7 text documents that contain the toy documents with the materials for this session in the racó. The steps to follow are:

- create an empty index
- open each text document in the `toy-docs` folder provided, read its contents and add it to the index as a new document; your index should contain 7 documents after this
- use the `termvectors` function to obtain term counts, generate Boolean and tf-idf matrices based on these counts.
- double check that your results coincide with the numbers in theory slides

For this toy corpus, you may build dense Boolean and TF-IDF matrices (e.g., using NumPy arrays).  Note, however, that for real datasets it would be necessary to store them as sparse matrices (e.g. in compressed row store format), which would use far less memory.

In [16]:
# Paso 1: Crear el índice para el ejercicio 5
from elasticsearch_dsl import Index
import os

# Crear el índice ex5
toy_index = Index('ex5', using=client)

# Eliminar el índice si ya existe (para empezar limpio)
if toy_index.exists():
    toy_index.delete()
    print('Índice ex5 eliminado (existía previamente)')

# Crear el nuevo índice
toy_index.create()
print('Índice ex5 creado exitosamente')

# Paso 2: Leer y cargar todos los documentos de toy-docs
toy_docs_path = 'toy-docs'
documents = []

# Leer los 7 documentos
for i in range(1, 8):
    filename = f'd{i}.txt'
    filepath = os.path.join(toy_docs_path, filename)
    
    with open(filepath, 'r') as file:
        content = file.read().strip()
        documents.append(content)
        print(f'{filename}: "{content}"')

print(f'\nTotal documentos leídos: {len(documents)}')

# Paso 3: Indexar los documentos en Elasticsearch
for i, doc_content in enumerate(documents, 1):
    doc_id = f'd{i}'
    client.index(
        index='ex5', 
        id=doc_id,
        document={'text': doc_content}
    )
    print(f'Documento {doc_id} indexado: "{doc_content}"')

# Refrescar el índice para asegurar que todos los documentos estén disponibles
client.indices.refresh(index='ex5')
print('\nTodos los documentos han sido indexados exitosamente en el índice ex5!')

# Verificar que tenemos 7 documentos
resp = client.search(index="ex5", query={"match_all": {}})
print(f"Total documentos en el índice: {resp['hits']['total']['value']}")

Índice ex5 creado exitosamente
d1.txt: "one three"
d2.txt: "two two three"
d3.txt: "one three four five five five"
d4.txt: "one two two two two three six six"
d5.txt: "three four four four six"
d6.txt: "three three three six six"
d7.txt: "four five"

Total documentos leídos: 7
Documento d1 indexado: "one three"
Documento d2 indexado: "two two three"
Documento d3 indexado: "one three four five five five"
Documento d4 indexado: "one two two two two three six six"
Documento d5 indexado: "three four four four six"
Documento d6 indexado: "three three three six six"
Documento d7 indexado: "four five"

Todos los documentos han sido indexados exitosamente en el índice ex5!
Total documentos en el índice: 7


In [15]:
# Paso 4: Extraer term vectors y preparar datos para las matrices
from elasticsearch.helpers import scan
import numpy as np
import pandas as pd
from collections import defaultdict

print("=== EXTRAYENDO TERM VECTORS ===")

# Obtener todos los documentos con sus IDs
documents_data = {}
doc_ids = []

# Escanear todos los documentos
resp = client.search(index="ex5", query={"match_all": {}}, size=10)
for hit in resp['hits']['hits']:
    doc_id = hit['_id']
    doc_ids.append(doc_id)
    documents_data[doc_id] = hit['_source']['text']

print(f"Documentos encontrados: {doc_ids}")

# Extraer term vectors de cada documento
term_frequencies = {}  # {doc_id: {term: freq}}
all_terms = set()      # conjunto de todos los términos únicos
term_doc_freq = defaultdict(int)  # cuántos documentos contienen cada término

for doc_id in doc_ids:
    print(f"\nProcesando {doc_id}: '{documents_data[doc_id]}'")
    
    # Obtener term vectors
    tv = client.termvectors(
        index='ex5', 
        id=doc_id, 
        fields=['text'], 
        term_statistics=True, 
        positions=False
    )
    
    if 'text' in tv['term_vectors']:
        terms_in_doc = tv['term_vectors']['text']['terms']
        term_frequencies[doc_id] = {}
        
        for term, term_info in terms_in_doc.items():
            freq = term_info['term_freq']
            doc_freq = term_info['doc_freq']  # en cuántos documentos aparece el término
            
            term_frequencies[doc_id][term] = freq
            all_terms.add(term)
            term_doc_freq[term] = doc_freq
            
            print(f"  {term}: freq={freq}, doc_freq={doc_freq}")

print(f"\nTérminos únicos encontrados: {sorted(all_terms)}")
print(f"Total de términos únicos: {len(all_terms)}")

# Ordenar términos y documentos para consistencia
sorted_terms = sorted(all_terms)
sorted_doc_ids = sorted(doc_ids)

print(f"\nDocumentos ordenados: {sorted_doc_ids}")
print(f"Términos ordenados: {sorted_terms}")

=== EXTRAYENDO TERM VECTORS ===
Documentos encontrados: ['d1', 'd2', 'd3', 'd4', 'd5', 'd6', 'd7']

Procesando d1: 'one three'
  one: freq=1, doc_freq=3
  three: freq=1, doc_freq=6

Procesando d2: 'two two three'
  three: freq=1, doc_freq=6
  two: freq=2, doc_freq=2

Procesando d3: 'one three four five five five'
  five: freq=3, doc_freq=2
  four: freq=1, doc_freq=3
  one: freq=1, doc_freq=3
  three: freq=1, doc_freq=6

Procesando d4: 'one two two two two three six six'
  one: freq=1, doc_freq=3
  six: freq=2, doc_freq=3
  three: freq=1, doc_freq=6
  two: freq=4, doc_freq=2

Procesando d5: 'three four four four six'
  four: freq=3, doc_freq=3
  six: freq=1, doc_freq=3
  three: freq=1, doc_freq=6

Procesando d6: 'three three three six six'
  six: freq=2, doc_freq=3
  three: freq=3, doc_freq=6

Procesando d7: 'four five'
  five: freq=1, doc_freq=2
  four: freq=1, doc_freq=3

Términos únicos encontrados: ['five', 'four', 'one', 'six', 'three', 'two']
Total de términos únicos: 6

Documento

In [16]:
# Paso 5: Generar la Matriz Boolean
print("=== GENERANDO MATRIZ BOOLEAN ===")

# Crear matriz boolean (documentos x términos)
boolean_matrix = np.zeros((len(sorted_doc_ids), len(sorted_terms)), dtype=int)

for i, doc_id in enumerate(sorted_doc_ids):
    for j, term in enumerate(sorted_terms):
        # Si el término aparece en el documento, poner 1, sino 0
        if doc_id in term_frequencies and term in term_frequencies[doc_id]:
            boolean_matrix[i, j] = 1

# Crear DataFrame para mejor visualización
boolean_df = pd.DataFrame(
    boolean_matrix, 
    index=sorted_doc_ids, 
    columns=sorted_terms
)

print("Matriz Boolean (documentos x términos):")
print(boolean_df)

# Mostrar también los documentos originales para referencia
print("\n=== DOCUMENTOS ORIGINALES ===")
for doc_id in sorted_doc_ids:
    print(f"{doc_id}: '{documents_data[doc_id]}'")

=== GENERANDO MATRIZ BOOLEAN ===
Matriz Boolean (documentos x términos):
    five  four  one  six  three  two
d1     0     0    1    0      1    0
d2     0     0    0    0      1    1
d3     1     1    1    0      1    0
d4     0     0    1    1      1    1
d5     0     1    0    1      1    0
d6     0     0    0    1      1    0
d7     1     1    0    0      0    0

=== DOCUMENTOS ORIGINALES ===
d1: 'one three'
d2: 'two two three'
d3: 'one three four five five five'
d4: 'one two two two two three six six'
d5: 'three four four four six'
d6: 'three three three six six'
d7: 'four five'


In [17]:
# Paso 6: Generar la Matriz TF-IDF
import math

print("=== GENERANDO MATRIZ TF-IDF ===")

# Número total de documentos
N = len(sorted_doc_ids)
print(f"Número total de documentos (N): {N}")

# Crear matriz TF-IDF (documentos x términos)
tfidf_matrix = np.zeros((len(sorted_doc_ids), len(sorted_terms)))

print("\nCalculando TF-IDF para cada término en cada documento:")

for i, doc_id in enumerate(sorted_doc_ids):
    print(f"\nDocumento {doc_id}: '{documents_data[doc_id]}'")
    
    for j, term in enumerate(sorted_terms):
        # TF (Term Frequency) - frecuencia del término en el documento
        tf = 0
        if doc_id in term_frequencies and term in term_frequencies[doc_id]:
            tf = term_frequencies[doc_id][term]
        
        # DF (Document Frequency) - en cuántos documentos aparece el término
        df = term_doc_freq[term]
        
        # IDF (Inverse Document Frequency)
        idf = math.log(N / df) if df > 0 else 0
        
        # TF-IDF = TF * IDF
        tfidf = tf * idf
        tfidf_matrix[i, j] = tfidf
        
        if tf > 0:  # Solo mostrar términos que aparecen en el documento
            print(f"  {term}: TF={tf}, DF={df}, IDF={idf:.3f}, TF-IDF={tfidf:.3f}")

# Crear DataFrame para mejor visualización
tfidf_df = pd.DataFrame(
    tfidf_matrix, 
    index=sorted_doc_ids, 
    columns=sorted_terms
)

print(f"\nMatriz TF-IDF (documentos x términos):")
print(tfidf_df.round(3))

# Mostrar estadísticas de los términos
print(f"\n=== ESTADÍSTICAS DE TÉRMINOS ===")
for term in sorted_terms:
    df = term_doc_freq[term]
    idf = math.log(N / df)
    print(f"{term}: aparece en {df}/{N} documentos, IDF = log({N}/{df}) = {idf:.3f}")

=== GENERANDO MATRIZ TF-IDF ===
Número total de documentos (N): 7

Calculando TF-IDF para cada término en cada documento:

Documento d1: 'one three'
  one: TF=1, DF=3, IDF=0.847, TF-IDF=0.847
  three: TF=1, DF=6, IDF=0.154, TF-IDF=0.154

Documento d2: 'two two three'
  three: TF=1, DF=6, IDF=0.154, TF-IDF=0.154
  two: TF=2, DF=2, IDF=1.253, TF-IDF=2.506

Documento d3: 'one three four five five five'
  five: TF=3, DF=2, IDF=1.253, TF-IDF=3.758
  four: TF=1, DF=3, IDF=0.847, TF-IDF=0.847
  one: TF=1, DF=3, IDF=0.847, TF-IDF=0.847
  three: TF=1, DF=6, IDF=0.154, TF-IDF=0.154

Documento d4: 'one two two two two three six six'
  one: TF=1, DF=3, IDF=0.847, TF-IDF=0.847
  six: TF=2, DF=3, IDF=0.847, TF-IDF=1.695
  three: TF=1, DF=6, IDF=0.154, TF-IDF=0.154
  two: TF=4, DF=2, IDF=1.253, TF-IDF=5.011

Documento d5: 'three four four four six'
  four: TF=3, DF=3, IDF=0.847, TF-IDF=2.542
  six: TF=1, DF=3, IDF=0.847, TF-IDF=0.847
  three: TF=1, DF=6, IDF=0.154, TF-IDF=0.154

Documento d6: 'three 

In [18]:
# Paso 7: Resumen y Verificación de Resultados
print("=== RESUMEN FINAL ===")

print("\n1. CORPUS TOY UTILIZADO:")
for i, doc_id in enumerate(sorted_doc_ids, 1):
    print(f"   Documento {i} ({doc_id}): '{documents_data[doc_id]}'")

print(f"\n2. VOCABULARIO EXTRAÍDO:")
print(f"   Términos únicos: {sorted_terms}")
print(f"   Total términos: {len(sorted_terms)}")

print(f"\n3. MATRIZ BOOLEAN ({len(sorted_doc_ids)} documentos × {len(sorted_terms)} términos):")
print(boolean_df)

print(f"\n4. MATRIZ TF-IDF ({len(sorted_doc_ids)} documentos × {len(sorted_terms)} términos):")
print(tfidf_df.round(3))

# Verificación adicional: mostrar frecuencias de términos por documento
print(f"\n5. VERIFICACIÓN - FRECUENCIAS DE TÉRMINOS:")
for doc_id in sorted_doc_ids:
    print(f"\n   {doc_id}: '{documents_data[doc_id]}'")
    if doc_id in term_frequencies:
        for term, freq in sorted(term_frequencies[doc_id].items()):
            print(f"     '{term}': {freq} veces")
    else:
        print("     (sin términos indexados)")

print(f"\n6. INFORMACIÓN ADICIONAL:")
print(f"   - Total documentos procesados: {len(sorted_doc_ids)}")
print(f"   - Total términos únicos: {len(sorted_terms)}")
print(f"   - Dimensión matriz Boolean: {boolean_matrix.shape}")
print(f"   - Dimensión matriz TF-IDF: {tfidf_matrix.shape}")

print(f"\n✅ EJERCICIO COMPLETADO EXITOSAMENTE!")
print(f"   Las matrices Boolean y TF-IDF han sido generadas correctamente.")
print(f"   Compara estos resultados con las diapositivas de teoría para verificar.")

=== RESUMEN FINAL ===

1. CORPUS TOY UTILIZADO:
   Documento 1 (d1): 'one three'
   Documento 2 (d2): 'two two three'
   Documento 3 (d3): 'one three four five five five'
   Documento 4 (d4): 'one two two two two three six six'
   Documento 5 (d5): 'three four four four six'
   Documento 6 (d6): 'three three three six six'
   Documento 7 (d7): 'four five'

2. VOCABULARIO EXTRAÍDO:
   Términos únicos: ['five', 'four', 'one', 'six', 'three', 'two']
   Total términos: 6

3. MATRIZ BOOLEAN (7 documentos × 6 términos):
    five  four  one  six  three  two
d1     0     0    1    0      1    0
d2     0     0    0    0      1    1
d3     1     1    1    0      1    0
d4     0     0    1    1      1    1
d5     0     1    0    1      1    0
d6     0     0    0    1      1    0
d7     1     1    0    0      0    0

4. MATRIZ TF-IDF (7 documentos × 6 términos):
     five   four    one    six  three    two
d1  0.000  0.000  0.847  0.000  0.154  0.000
d2  0.000  0.000  0.000  0.000  0.154  2.506
d3

In [19]:
# Matrices Boolean y TF-IDF
print("MATRIZ BOOLEAN:")
print(boolean_df)
print("\nMATRIZ TF-IDF:")
print(tfidf_df.round(3))

MATRIZ BOOLEAN:
    five  four  one  six  three  two
d1     0     0    1    0      1    0
d2     0     0    0    0      1    1
d3     1     1    1    0      1    0
d4     0     0    1    1      1    1
d5     0     1    0    1      1    0
d6     0     0    0    1      1    0
d7     1     1    0    0      0    0

MATRIZ TF-IDF:
     five   four    one    six  three    two
d1  0.000  0.000  0.847  0.000  0.154  0.000
d2  0.000  0.000  0.000  0.000  0.154  2.506
d3  3.758  0.847  0.847  0.000  0.154  0.000
d4  0.000  0.000  0.847  1.695  0.154  5.011
d5  0.000  2.542  0.000  0.847  0.154  0.000
d6  0.000  0.000  0.000  1.695  0.462  0.000
d7  1.253  0.847  0.000  0.000  0.000  0.000


In [20]:
import os
from elasticsearch_dsl import Index
from elasticsearch.helpers import scan
from collections import Counter
import numpy as np

# Define the path to the toy documents folder
toy_docs_folder = 'toy-docs' # Assuming 'toy-docs' is in the same directory as the notebook

# 1. Create an empty index
index_name = 'toy_corpus'
if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)
index = Index(index_name, using=client)
index.create()

# 2. Index each text document
documents = []
for filename in os.listdir(toy_docs_folder):
    if filename.endswith(".txt"):
        filepath = os.path.join(toy_docs_folder, filename)
        with open(filepath, 'r') as f:
            content = f.read()
            documents.append({'filename': filename, 'text': content})
            client.index(index=index_name, document={'filename': filename, 'text': content})

client.indices.refresh(index=index_name)
print(f"Indexed {len(documents)} documents in index '{index_name}'")

# 3. Use termvectors to obtain term counts and generate matrices
# Get all terms and their document frequencies (DF)
term_df = Counter()
all_terms = set()
sc = scan(client, index=index_name, query={"query" : {"match_all": {}}})
for s in sc:
    tv = client.termvectors(index=index_name, id=s['_id'], fields=['text'], term_statistics=True, positions=False)
    if 'text' in tv['term_vectors']:
        for t in tv['term_vectors']['text']['terms']:
            term = t
            all_terms.add(term)
            # term_df.update({term: tv['term_vectors']['text']['terms'][term]['doc_freq']}) # This is document frequency

# Re-scan to get term frequencies per document
term_list = sorted(list(all_terms))
num_docs = len(documents)
num_terms = len(term_list)

# Initialize matrices
boolean_matrix = np.zeros((num_docs, num_terms), dtype=int)
tf_matrix = np.zeros((num_docs, num_terms), dtype=int)

doc_id_map = {doc['filename']: i for i, doc in enumerate(documents)}

sc = scan(client, index=index_name, query={"query" : {"match_all": {}}})
for s in sc:
    doc_filename = s['_source']['filename']
    doc_idx = doc_id_map[doc_filename]
    tv = client.termvectors(index=index_name, id=s['_id'], fields=['text'], term_statistics=True, positions=False)

    if 'text' in tv['term_vectors']:
        for t in tv['term_vectors']['text']['terms']:
            term = t
            term_idx = term_list.index(term)
            term_freq = tv['term_vectors']['text']['terms'][term]['term_freq']

            # Update Boolean matrix
            boolean_matrix[doc_idx, term_idx] = 1

            # Update TF matrix
            tf_matrix[doc_idx, term_idx] = term_freq

# Calculate IDF
# Need document frequencies for IDF calculation
term_doc_freq = Counter()
sc_df = scan(client, index=index_name, query={"query" : {"match_all": {}}})
for s_df in sc_df:
    tv_df = client.termvectors(index=index_name, id=s_df['_id'], fields=['text'], term_statistics=True, positions=False)
    if 'text' in tv_df['term_vectors']:
         for t_df in tv_df['term_vectors']['text']['terms']:
             term_doc_freq.update({t_df:1}) # Count documents containing the term

idf_vector = np.zeros(num_terms)
N = num_docs
for i, term in enumerate(term_list):
    df = term_doc_freq.get(term, 0)
    if df > 0:
        idf_vector[i] = np.log10(N / df)

# Calculate TF-IDF matrix
tfidf_matrix = tf_matrix * idf_vector

print("\nTerms:")
print(term_list)

print("\nBoolean Matrix:")
print(boolean_matrix)

print("\nTF Matrix:")
print(tf_matrix)

print("\nIDF Vector:")
print(idf_vector)

print("\nTF-IDF Matrix:")
print(tfidf_matrix)

Indexed 7 documents in index 'toy_corpus'

Terms:
['five', 'four', 'one', 'six', 'three', 'two']

Boolean Matrix:
[[0 0 1 1 1 1]
 [0 0 0 1 1 0]
 [1 1 0 0 0 0]
 [0 0 0 0 1 1]
 [0 1 0 1 1 0]
 [1 1 1 0 1 0]
 [0 0 1 0 1 0]]

TF Matrix:
[[0 0 1 2 1 4]
 [0 0 0 2 3 0]
 [1 1 0 0 0 0]
 [0 0 0 0 1 2]
 [0 3 0 1 1 0]
 [3 1 1 0 1 0]
 [0 0 1 0 1 0]]

IDF Vector:
[0.54406804 0.36797679 0.36797679 0.36797679 0.06694679 0.54406804]

TF-IDF Matrix:
[[0.         0.         0.36797679 0.73595357 0.06694679 2.17627218]
 [0.         0.         0.         0.73595357 0.20084037 0.        ]
 [0.54406804 0.36797679 0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.06694679 1.08813609]
 [0.         1.10393036 0.         0.36797679 0.06694679 0.        ]
 [1.63220413 0.36797679 0.36797679 0.         0.06694679 0.        ]
 [0.         0.         0.36797679 0.         0.06694679 0.        ]]


## 6. Cleanup

Finally, we remove the test index..

In [12]:
index.delete()

ObjectApiResponse({'acknowledged': True})